# Device Location Analysis

This notebook resolves the latest serving cell for each device in a fleet and enriches those cells with coordinates from Unwired Labs LocationAPI (OpenCellID data).

Steps:
1. Configure API keys and credentials.
2. Fetch the device list for the device group API key (or use DEVICE_IDS).
3. Query Ops API status (no direct database access).
4. Extract cell tower identifiers.
5. Enrich with LocationAPI and save a CSV next to the notebook.

## Imports

In [50]:
import os
import json
import time
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urlencode

import pandas as pd
import requests

## Configuration

In [51]:
# Public API key for the target device group
DEVICE_GROUP_API_KEY: str = ""

# Ops API key for device status
OPS_API_KEY: str = ""


# User Apps API credentials (not used for Ops API status)
UA_USERNAME: str = ""
UA_PASSWORD: str = ""

# Unwired Labs LocationAPI token (OpenCellID data)
OPENCELLID_API_KEY: str = ""

In [52]:
API_REQUESTS_PER_SEC: float = float(os.getenv("API_REQUESTS_PER_SEC", "2"))

OUTPUT_FILENAME: str = "device_location_enriched.csv"

OPS_API_BASE_URL: str = "https://api.ops.wattwatchers.net"

# If you want to analyse a subset of devices, enter the device ids inside the brackets ([])
# Example: ["DD1234567890", "DD2345678901"]
DEVICE_IDS: list[str] = []

In [53]:
# Secure prompt (only used if API keys are empty)
from getpass import getpass

if not DEVICE_GROUP_API_KEY:
    DEVICE_GROUP_API_KEY = getpass("Device group API key: ").strip()
if not OPS_API_KEY:
    OPS_API_KEY = getpass("Ops API key: ").strip()
if not OPENCELLID_API_KEY:
    OPENCELLID_API_KEY = getpass("OpenCellID API key: ").strip()

## Logging and helpers

In [54]:
def get_logger(logging_level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger("device-location")
    logger.setLevel(logging_level)
    if logger.hasHandlers():
        return logger
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger


logger = get_logger()


def convert_eci_to_cellid(eci: int) -> int:
    try:
        return int(eci) & 0xFF
    except Exception:
        return int(eci)


def convert_eci_to_enb(eci: int) -> int | None:
    try:
        return int(eci) >> 8
    except Exception:
        return None

## Public API client (device list)

In [55]:
PUBLIC_API_BASE_URL = "https://api-v3.wattwatchers.com.au"

def fetch_device_ids(api_key: str) -> list[str]:
    if not api_key:
        raise ValueError("DEVICE_GROUP_API_KEY is required when DEVICE_IDS is empty.")
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    response = requests.get(f"{PUBLIC_API_BASE_URL}/devices", headers=headers, timeout=30)
    response.raise_for_status()
    device_ids = response.json()
    if not isinstance(device_ids, list):
        raise ValueError("Unexpected devices response from Public API.")
    return [str(device_id) for device_id in device_ids]


def resolve_device_ids() -> list[str]:
    if DEVICE_IDS:
        return [str(device_id).strip() for device_id in DEVICE_IDS if str(device_id).strip()]
    return fetch_device_ids(DEVICE_GROUP_API_KEY)

## Ops API status (no direct DB access)

In [56]:
def _ops_headers() -> dict[str, str]:
    if not OPS_API_KEY:
        raise ValueError("OPS_API_KEY is required.")
    return {
        "Authorization": f"Bearer {OPS_API_KEY}",
        "Content-Type": "application/json",
    }


def fetch_ops_status(device_id: str, raw: bool = True) -> dict[str, Any] | None:
    params = {"raw": "true"} if raw else {}
    query = f"?{urlencode(params)}" if params else ""
    url = f"{OPS_API_BASE_URL}/devices/{device_id}/status{query}"
    try:
        response = requests.get(url, headers=_ops_headers(), timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        logger.error(f"Ops status failed for {device_id}: {exc}")
        return None


def fetch_latest_status_via_ops(device_ids: Iterable[str]) -> dict[str, dict[str, Any]]:
    status_map: dict[str, dict[str, Any]] = {}
    for device_id in device_ids:
        status_payload = fetch_ops_status(device_id, raw=True)
        if status_payload:
            status_map[str(device_id)] = status_payload
    return status_map

## Parse cell tower identifiers

In [57]:
def _to_int(value: Any) -> int | None:
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        try:
            return int(text, 16)
        except ValueError:
            return None


def _radio_type_from_mode(mode: str | None) -> str | None:
    if not mode:
        return None
    mode_upper = mode.upper()
    if mode_upper == "4G":
        return "LTE"
    if mode_upper == "3G":
        return "UMTS"
    if mode_upper == "2G":
        return "GSM"
    return None


def parse_serving_cell(serving_cell: str) -> dict[str, Any] | None:
    if not serving_cell:
        return None
    parts = [part.strip() for part in serving_cell.split(",") if part.strip()]
    if len(parts) < 8:
        return None
    mode = parts[0].upper()
    radio_type = _radio_type_from_mode(mode)
    mcc = _to_int(parts[6]) if len(parts) > 6 else None
    mnc = _to_int(parts[7]) if len(parts) > 7 else None
    tac = _to_int(parts[8]) if len(parts) > 8 else None
    eci = _to_int(parts[9]) if len(parts) > 9 else None
    if mcc is None or mnc is None:
        return None
    return {"mcc": mcc, "mnc": mnc, "tac": tac, "eci": eci, "radioType": radio_type}


def _parse_network_id(network_id: Any) -> tuple[int | None, int | None]:
    if network_id is None:
        return None, None
    text = str(network_id).strip()
    if not text.isdigit() or len(text) < 4:
        return None, None
    mcc = _to_int(text[:3])
    mnc = _to_int(text[3:])
    return mcc, mnc


def _extract_cell_from_ops(status_payload: dict[str, Any]) -> dict[str, Any] | None:
    if not isinstance(status_payload, dict):
        return None
    raw = status_payload.get("raw") if isinstance(status_payload.get("raw"), dict) else None
    if raw:
        serving_cell = raw.get("_serving_cell") or raw.get("serving_cell")
        parsed = parse_serving_cell(serving_cell)
        if parsed:
            return parsed
    serving_cell = status_payload.get("_serving_cell") or status_payload.get("serving_cell")
    parsed = parse_serving_cell(serving_cell)
    if parsed:
        return parsed
    cell_fields = {
        "mcc": status_payload.get("cellMcc") or status_payload.get("mcc"),
        "mnc": status_payload.get("cellMnc") or status_payload.get("mnc"),
        "tac": status_payload.get("cellTac") or status_payload.get("tac"),
        "eci": status_payload.get("cellEci") or status_payload.get("eci"),
        "radioType": status_payload.get("cellRadioType") or status_payload.get("radioType"),
    }
    if any(value is not None for value in cell_fields.values()):
        return cell_fields
    mcc, mnc = _parse_network_id(status_payload.get("cellNetworkOperator"))
    if mcc or mnc:
        return {"mcc": mcc, "mnc": mnc, "tac": None, "eci": None, "radioType": None}
    return None


def extract_cell_info(raw_status: dict[str, Any]) -> dict[str, Any] | None:
    if not raw_status:
        return None
    ops_cell = _extract_cell_from_ops(raw_status)
    if isinstance(ops_cell, dict):
        return {
            "mcc": _to_int(ops_cell.get("mcc")),
            "mnc": _to_int(ops_cell.get("mnc")),
            "tac": _to_int(ops_cell.get("tac")),
            "eci": _to_int(ops_cell.get("eci")),
            "radioType": ops_cell.get("radioType"),
        }
    for key in ["cell", "cellInfo", "cell_info"]:
        cell_payload = raw_status.get(key)
        if isinstance(cell_payload, dict):
            return {
                "mcc": _to_int(cell_payload.get("mcc")),
                "mnc": _to_int(cell_payload.get("mnc")),
                "tac": _to_int(cell_payload.get("tac")),
                "eci": _to_int(cell_payload.get("eci")),
                "radioType": cell_payload.get("radioType") or cell_payload.get("radio"),
            }
    serving_cell = raw_status.get("_serving_cell") or raw_status.get("serving_cell")
    return parse_serving_cell(serving_cell)

## OpenCellID client

In [58]:
@dataclass
class AppConfig:
    opencellid_api_key: str
    requests_per_sec: float = 2


class OpenCellIdApiClient:
    def __init__(self, api_key: str, base_url: str = "https://us1.unwiredlabs.com/v2/process", requests_per_sec_max: float = 2):
        self.api_key = api_key
        self.base_url = base_url
        self.min_request_interval = 1.0 / requests_per_sec_max if requests_per_sec_max else 0
        self.last_request_time = 0.0

    def _rate_limit(self) -> None:
        if self.min_request_interval > 0:
            elapsed = time.time() - self.last_request_time
            if elapsed < self.min_request_interval:
                time.sleep(self.min_request_interval - elapsed)
        self.last_request_time = time.time()

    def _normalize_radio(self, radio_type: str | None) -> str | None:
        if not radio_type:
            return None
        text = str(radio_type).strip().lower()
        mapping = {
            "lte": "lte",
            "4g": "lte",
            "umts": "umts",
            "3g": "umts",
            "gsm": "gsm",
            "2g": "gsm",
            "nbiot": "nbiot",
            "cat-m": "lte",
            "catm": "lte",
            "nr": "nr",
            "5g": "nr",
        }
        return mapping.get(text, text)

    def get_cell_location(
        self,
        mcc: int,
        mnc: int,
        lac_or_tac: int,
        cellid_or_eci: int,
        radio_type: str | None = None,
    ) -> tuple[dict[str, Any] | None, str | None]:
        self._rate_limit()
        payload: dict[str, Any] = {
            "token": self.api_key,
            "mcc": mcc,
            "mnc": mnc,
            "cells": [
                {
                    "lac": lac_or_tac,
                    "cid": cellid_or_eci,
                }
            ],
            "address": 1,
        }
        radio = self._normalize_radio(radio_type)
        if radio:
            payload["radio"] = radio
        try:
            response = requests.post(self.base_url, json=payload, timeout=30)
            if response.status_code != 200:
                return None, f"API request failed with status {response.status_code}: {response.text}"
            data = response.json()
            if data.get("status") == "ok" and "lon" in data and "lat" in data:
                return data, None
            if data.get("status") == "error":
                return None, data.get("message") or "API error"
            return None, "No location data in response"
        except Exception as exc:
            return None, f"Error in API request: {exc}"

## Build device cell dataset

In [59]:
device_ids = resolve_device_ids()
logger.info(f"Resolved {len(device_ids)} device IDs")

raw_status_by_device = fetch_latest_status_via_ops(device_ids)
logger.info(f"Loaded Ops status for {len(raw_status_by_device)} devices")

records: list[dict[str, Any]] = []
for device_id in device_ids:
    raw_status = raw_status_by_device.get(device_id)
    cell_info = extract_cell_info(raw_status or {})
    if not cell_info:
        records.append({
            "device_id": device_id,
            "site": None,
            "mcc": None,
            "mnc": None,
            "tac": None,
            "eci": None,
            "radioType": None,
            "enb_id": None,
            "cell_id": None,
        })
        continue
    eci = cell_info.get("eci")
    records.append({
        "device_id": device_id,
        "site": None,
        "mcc": cell_info.get("mcc"),
        "mnc": cell_info.get("mnc"),
        "tac": cell_info.get("tac"),
        "eci": eci,
        "radioType": cell_info.get("radioType"),
        "enb_id": convert_eci_to_enb(eci) if eci is not None else None,
        "cell_id": convert_eci_to_cellid(eci) if eci is not None else None,
    })

df_cells = pd.DataFrame.from_records(records)
df_cells

ValueError: DEVICE_GROUP_API_KEY is required when DEVICE_IDS is empty.

In [ ]:
# Quick LocationAPI sanity check using locations.csv (if present)
csv_path = Path.cwd() / "device_location_analysis" / "locations.csv"
if not csv_path.exists():
    csv_path = Path.cwd() / "locations.csv"
if csv_path.exists():
    test_df = pd.read_csv(csv_path)
    test_row = test_df.dropna(subset=["mcc", "mnc", "tac", "eci"]).head(1)
    if not test_row.empty:
        row = test_row.iloc[0]
        test_mcc = int(row["mcc"])
        test_mnc = int(row["mnc"])
        test_tac = int(row["tac"])
        test_eci = int(row["eci"])
        test_radio = "lte"
        print(f"Testing LocationAPI with mcc={test_mcc}, mnc={test_mnc}, tac={test_tac}, eci={test_eci}")
        test_client = OpenCellIdApiClient(OPENCELLID_API_KEY, requests_per_sec_max=API_REQUESTS_PER_SEC)
        location, error = test_client.get_cell_location(test_mcc, test_mnc, test_tac, test_eci, radio_type=test_radio)
        print("Error:", error)
        print("Response:", location)
    else:
        print("No valid mcc/mnc/tac/eci rows found in locations.csv")
else:
    print("locations.csv not found; skipping LocationAPI sanity check")

Testing LocationAPI with mcc=505, mnc=2, tac=58501, eci=25188874
Error: None
Response: {'status': 'ok', 'balance': 4999, 'lat': -34.89714, 'lon': 138.66676, 'accuracy': 2317}


In [ ]:
# Debug: inspect Ops status payload for one device
sample_device = device_ids[0] if device_ids else None
if sample_device:
    payload = raw_status_by_device.get(sample_device, {})
    print(f"Sample device: {sample_device}")
    print(f"Top-level keys: {sorted(payload.keys())}")
    raw = payload.get("raw", {}) if isinstance(payload, dict) else {}
    print(f"Raw keys: {sorted(raw.keys())}" if isinstance(raw, dict) else "Raw not dict")

Sample device: DD05335138513
Top-level keys: ['_IMSI', '_cell_quality', '_freq60', '_le_req_time', '_meta::active_transport', '_meta::tag', '_network_in_time', '_network_operator', '_option', '_powerdown_seqnum', '_serving_cell', '_signal_quality', '_sim_id_s', '_temperature', '_time_source', 'apn', 'boot_count', 'cal_igain', 'cal_vgain', 'comms_watchdog', 'feature_leadbeater', 'is_rogowski', 'leadbeater_mask_added', 'modbus_read_period', 'nturns', 'num_channels', 'num_conductors', 'phase_swap', 'polarity', 'rx_timestamp', 'se_ext', 'se_ext_enabled', 'se_mask', 'serial', 'server1_name', 'server2_name', 'server3_name', 'server_available', 'short_energy_report_divider', 'switch_3_nocomms_state', 'switch_3_nocomms_timeout', 'test_config', 'time_server', 'user_rat', 'version', 'vnom']
Raw keys: []


## Enrich with OpenCellID and save CSV

In [ ]:
if not OPENCELLID_API_KEY:
    raise ValueError("OPENCELLID_API_KEY is required.")

config = AppConfig(opencellid_api_key=OPENCELLID_API_KEY, requests_per_sec=API_REQUESTS_PER_SEC)
client = OpenCellIdApiClient(config.opencellid_api_key, requests_per_sec_max=config.requests_per_sec)

df_cells["longitude"] = None
df_cells["latitude"] = None
df_cells["accuracy"] = None
df_cells["lookup_status"] = None

unique_cells = (
    df_cells[["mcc", "mnc", "tac", "eci", "radioType"]]
    .dropna(subset=["mcc", "mnc", "tac", "eci"])
    .drop_duplicates()
)
location_cache: dict[str, dict[str, Any]] = {}

for _, row in unique_cells.iterrows():
    mcc = int(row["mcc"])
    mnc = int(row["mnc"])
    tac = int(row["tac"])
    eci = int(row["eci"])
    radio_type = row.get("radioType")
    cache_key = f"{mcc}-{mnc}-{tac}-{eci}-{radio_type or ''}"
    location, error = client.get_cell_location(mcc, mnc, tac, eci, radio_type=radio_type)
    if error:
        location_cache[cache_key] = {"error": error}
    else:
        location_cache[cache_key] = location or {"error": "not_found"}

for idx, row in df_cells.iterrows():
    if pd.isna(row["mcc"]) or pd.isna(row["mnc"]) or pd.isna(row["tac"]) or pd.isna(row["eci"]):
        df_cells.at[idx, "lookup_status"] = "missing_cell"
        continue
    radio_type = row.get("radioType")
    cache_key = f"{int(row['mcc'])}-{int(row['mnc'])}-{int(row['tac'])}-{int(row['eci'])}-{radio_type or ''}"
    location = location_cache.get(cache_key, {})
    if "error" in location:
        df_cells.at[idx, "lookup_status"] = location["error"]
        continue
    df_cells.at[idx, "longitude"] = location.get("lon")
    df_cells.at[idx, "latitude"] = location.get("lat")
    df_cells.at[idx, "accuracy"] = location.get("accuracy")
    df_cells.at[idx, "lookup_status"] = "found"

notebook_dir = Path.cwd()
if (notebook_dir / "device_location_analysis.ipynb").exists():
    pass
elif (notebook_dir / "device_location_analysis").exists():
    notebook_dir = notebook_dir / "device_location_analysis"
output_path = notebook_dir / OUTPUT_FILENAME
df_cells.to_csv(output_path, index=False)
logger.info(f"Saved {len(df_cells)} rows to {output_path}")
df_cells.head()

2026-04-30 11:41:56,704 INFO Saved 4 rows to /Users/kevinquigley/Library/CloudStorage/OneDrive-Personal/Documents/wattwatchers-Mac/le_completeness_analysis/device_location_analysis/device_location_enriched.csv


,device_id,site,mcc,mnc,tac,eci,radioType,enb_id,cell_id,longitude,latitude,accuracy,lookup_status
0,DD05335138513,None,505,2,58501,25188874,LTE,98394,10,138.66676,-34.89714,2317,found
1,DD45335175553,None,505,2,58501,25188914,LTE,98394,50,138.667024,-34.896378,2358,found
2,DD45335178107,None,505,1,32782,148995841,LTE,582015,1,138.68281,-34.90869,2462,found
3,DDD5335178031,None,505,1,32782,148995595,LTE,582014,11,138.679051,-34.90489,1622,found
